# 用 LangGraph 构建可解释、可拒答的静态进度 RAG

本 notebook 只围绕仓库现有的 **8 份输变电工程进度 TXT**。目标不是把简单查询包装得更复杂，而是回答两个工程问题：

1. 语义检索召回了相似文本，为什么还不能直接相信日期？
2. 怎样把工程消歧、完整任务记录、字段校验、引用和拒答做成可观察、可测试的控制流？

最终结论会由实际评测给出：单跳事实仍优先走直接 reliable pipeline；LangGraph 的主要价值是显式状态、条件分支、checkpoint 和调试能力，不会自动提高 retriever 的质量。

## 1. LangGraph 的设计理念，以及它不负责什么

**LangGraph 是低层 orchestration/runtime（编排与运行时）**。它位于 LangChain 的模型、Document、Retriever、Tool 等组件之上，用有向图组织长流程。

- **State（状态）**：每一步只读所需字段并返回增量更新，因而可以观察“问题被怎样解析、检索了几次、为什么拒答”。
- **Node（节点）**：把规范化、工程路由、候选召回、任务定位和 Evidence Gate 分成可独立测试的边界。
- **Conditional Edge（条件边）**：把回答、澄清、一次 fallback、拒答和冲突报告写成显式控制流。
- **Checkpointer（检查点）**：按 `thread_id` 保存一次 graph 的短期状态，便于恢复和排查；它不是跨用户长期知识记忆。

LangGraph **不是 retriever**，也不会把相似度变成事实正确率。`durable execution`、HITL、长期 memory 都是框架能力，但当前静态单次查询不需要全部开启。本项目选择有限状态图，而不是开放式 ReAct 循环：路径是已知的，检索最多两次，日期不交给 LLM 猜。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, Literal, TypedDict

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")


def find_repo_root(start: Path | None = None) -> Path:
    """从仓库根目录或 ZZworkbench 启动时都能定位资源。"""

    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        corpus = candidate / "knowledge" / "project_progress" / "texts"
        if corpus.is_dir() and (candidate / "ZZworkbench").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate the pipelines_rag repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from langchain_core.documents import Document
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

from ZZworkbench.project_progress_reliable import (
    ProjectProgressKnowledgeBase,
    QueryIntent,
    evaluate_reliable_lookup,
    normalize_lookup_text,
)
from ZZworkbench.rag_langchain.text_retrieval import (
    DEFAULT_CORPUS_ROOT,
    DEFAULT_INDEX_ROOT,
    ChunkConfig,
    EmbeddingConfig,
    RetrievalHit,
    build_embeddings,
    build_or_reuse_chroma,
    evaluate_retriever,
    load_eval_cases,
    load_txt_documents,
    open_chroma,
    retrieve,
    retrieve_hybrid,
    split_documents,
)

print("repository located:", REPO_ROOT.name == "pipelines_rag")

repository located: True


## 2. 固定语料与可靠证据合同

当前 v4 是小型、半结构化、强实体约束的数据：工程名和任务名比开放域语义更重要。共享模块先把完整任务段解析为 `ScheduleRecord`，再用同一套 Evidence Gate 服务直接 pipeline、LangGraph 和后面的 Deep Agents。

Character chunk 是否需要重做不能靠想象决定。下面先审计 541 条任务记录是否都能回指当前 63 个 chunk；若没有真实切断失败，本轮不重做索引。

In [2]:
chunk_config = ChunkConfig()
documents = load_txt_documents(DEFAULT_CORPUS_ROOT, version="v4")
chunks = split_documents(documents, chunk_config)
knowledge_base = ProjectProgressKnowledgeBase(documents, chunks)

audit = knowledge_base.audit()
stable_audit = {
    key: value
    for key, value in audit.items()
    if key != "repeated_task_names_across_sources"
}
stable_audit["repeated_task_name_count"] = len(
    audit["repeated_task_names_across_sources"]
)
print(json.dumps(stable_audit, ensure_ascii=False, indent=2))

assert audit["documents"] == 8
assert audit["chunks"] == 63
assert audit["records"] == 541
assert audit["records_without_chunk"] == 0

{
  "documents": 8,
  "chunks": 63,
  "records": 541,
  "records_without_chunk": 0,
  "records_without_start": 0,
  "records_without_end": 0,
  "records_without_duration": 39,
  "repeated_task_name_count": 28
}


In [3]:
embedding_config = EmbeddingConfig()
embeddings = build_embeddings(embedding_config)
index_result = build_or_reuse_chroma(
    documents,
    chunks,
    embeddings,
    embedding_config,
    persist_directory=DEFAULT_INDEX_ROOT,
    corpus_root=DEFAULT_CORPUS_ROOT,
    version="v4",
    chunk_config=chunk_config,
)
vector_store = open_chroma(DEFAULT_INDEX_ROOT, embeddings)

print(
    {
        "index_reused": index_result.reused,
        "stored_chunks": vector_store._collection.count(),
        "fingerprint_prefix": index_result.fingerprint[:12],
    }
)

{'index_reused': True, 'stored_chunks': 63, 'fingerprint_prefix': '6aedfff8739a'}


## 3. 先比较 A/B/C，不预设高级方案一定更好

三条路径使用同一份 v4 语料和同一批正例：

- **A dense-only**：语义召回基线，适合口语改写，但容易弱化工程名、编号和同名任务边界。
- **B hybrid**：工程 metadata route + 中文 n-gram BM25 + dense + weighted RRF。RRF 融合排名，不直接相加不可比的原始分数。
- **C reliable pipeline**：在 hybrid 候选上进一步做工程消歧、完整任务记录精确匹配、字段与引用校验；只有 `exact` 才回答。

相似度只表示候选排序信号，不是“日期正确概率”。特别是“施工准备”跨多个工程重复，没有工程名时最高相似 chunk 也不能替用户作选择。

In [4]:
eval_dir = REPO_ROOT / "knowledge" / "project_progress" / "evals"
positive_cases = load_eval_cases(eval_dir / "retrieval_v4.jsonl")
reliability_cases = load_eval_cases(eval_dir / "reliability_v4.jsonl")
all_cases = [*positive_cases, *reliability_cases]


def hybrid_candidates(query: str) -> list[RetrievalHit]:
    """固定候选预算；调用方不能任意放大 k。"""

    return retrieve_hybrid(
        vector_store,
        query,
        k=8,
        fetch_k=30,
        dense_weight=0.35,
    )


dense_report = evaluate_retriever(
    vector_store, positive_cases, k=4, strategy="dense"
)
hybrid_report = evaluate_retriever(
    vector_store,
    positive_cases,
    k=4,
    strategy="hybrid",
    fetch_k=20,
    dense_weight=0.35,
)
reliable_report = evaluate_reliable_lookup(
    knowledge_base,
    all_cases,
    retriever=hybrid_candidates,
)

comparison = {
    "dense-only": {
        "source_hit@4": dense_report["source_hit_at_k"],
        "term_hit@4": dense_report["term_hit_at_k"],
    },
    "hybrid": {
        "source_hit@4": hybrid_report["source_hit_at_k"],
        "term_hit@4": hybrid_report["term_hit_at_k"],
    },
    "reliable": {
        "cases": reliable_report["cases"],
        "status_accuracy": reliable_report["status_accuracy"],
        "source_accuracy": reliable_report["source_accuracy"],
        "requested_field_completeness": reliable_report["field_accuracy"],
        "pass_rate": reliable_report["pass_rate"],
    },
}
print(json.dumps(comparison, ensure_ascii=False, indent=2))

{
  "dense-only": {
    "source_hit@4": 0.5,
    "term_hit@4": 0.0
  },
  "hybrid": {
    "source_hit@4": 1.0,
    "term_hit@4": 1.0
  },
  "reliable": {
    "cases": 13,
    "status_accuracy": 1.0,
    "source_accuracy": 1.0,
    "requested_field_completeness": 1.0,
    "pass_rate": 1.0
  }
}


In [5]:
ambiguous_query = "施工准备什么时候完成？"
dense_ambiguous = retrieve(vector_store, ambiguous_query, k=4)
print("dense-only top sources:")
for hit in dense_ambiguous:
    print(hit.rank, hit.document.metadata.get("source_name"))

guarded = knowledge_base.lookup(
    ambiguous_query,
    retriever=hybrid_candidates,
)
print("Evidence Gate:", guarded.status)
print(guarded.answer)

dense-only top sources:
1 三虎输变电工程三级进度计划土建部分.txt
2 南溪三级进度.txt
3 110千伏节点计划-重点关注.txt
4 110千伏节点计划-重点关注.txt


Evidence Gate: ambiguous
任务“施工准备”出现在多个工程文档中，请补充工程范围：110kV黄金输变电工程三级进度计划.txt、三级进度计划-土建.txt、三虎输变电工程三级进度计划土建部分.txt、珠海110千伏江湾输变电工程施工进度计划（202.txt


## 4. QueryIntent：把自然语言变成可检查合同

首版不让 LLM 抽取规则已经能稳定识别的工程名和日期字段。规范化统一全半角、空白、`110kV/110千伏` 与已知词形；工程别名只来自这 8 份文档。`requested_fields` 决定 Evidence Gate 必须在同一条任务记录中找到哪些字段。

In [6]:
intent_examples = [
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "南溪旅游输变电工程的地基基础施工计划开始、完成和工期分别是什么？",
    "施工准备什么时候完成？",
]
for question in intent_examples:
    intent = knowledge_base.parse_query(question)
    print(
        json.dumps(
            {
                "original_query": intent.original_query,
                "normalized_query": intent.normalized_query,
                "project_hint": intent.project_hint,
                "task_hint": intent.task_hint,
                "requested_fields": intent.requested_fields,
                "query_type": intent.query_type,
                "diagnostics": intent.diagnostics,
            },
            ensure_ascii=False,
            indent=2,
        )
    )

{
  "original_query": "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
  "normalized_query": "珠海110千伏黄金输变电工程的主体结构封顶计划什么时候完成",
  "project_hint": "珠海110千伏黄金输变电工程工程进度计划横道图",
  "task_hint": "主体结构封顶",
  "requested_fields": [
    "end_date"
  ],
  "query_type": "single_lookup",
  "diagnostics": [
    "project route matched 1 source(s)"
  ]
}
{
  "original_query": "南溪旅游输变电工程的地基基础施工计划开始、完成和工期分别是什么？",
  "normalized_query": "南溪旅游输变电工程的地基基础施工计划开始完成和工期分别是什么",
  "project_hint": "珠海110千伏南溪（旅游）输变电工程施工进度计划横道图",
  "task_hint": "地基基础施工",
  "requested_fields": [
    "start_date",
    "end_date",
    "duration"
  ],
  "query_type": "single_lookup",
  "diagnostics": [
    "project route matched 1 source(s)"
  ]
}
{
  "original_query": "施工准备什么时候完成？",
  "normalized_query": "施工准备什么时候完成",
  "project_hint": null,
  "task_hint": "施工准备",
  "requested_fields": [
    "end_date"
  ],
  "query_type": "ambiguous",
  "diagnostics": [
    "no explicit project route"
  ]
}


## 5. Graph state：只放可序列化数据

`vector_store`、embedding model 和文件句柄是进程内依赖，由节点闭包只读使用；它们不进入 state。候选 `Document` 被转换为普通字典，checkpoint 因而可序列化。每个节点返回局部更新，不修改隐藏的全局状态。

控制流严格有界：默认一次 hybrid；仅 `not_found/insufficient` 允许一次 dense 补充召回，总 retrieval 次数不超过 2。`ambiguous` 不重搜，因为增加相似文本不能替代工程信息。

In [7]:
EvidenceStatus = Literal[
    "exact", "ambiguous", "not_found", "insufficient", "conflict"
]


class ReliableRAGState(TypedDict, total=False):
    question: str
    normalized_query: str
    intent: dict[str, Any]
    project_sources: list[str]
    candidate_hits: list[dict[str, Any]]
    retrieval_attempts: int
    fallback_used: bool
    retrieval_mode: str
    lookup_result: dict[str, Any]
    evidence_status: EvidenceStatus
    answer: str


def serialize_hit(hit: RetrievalHit) -> dict[str, Any]:
    document = hit.document
    return {
        "id": str(document.id or document.metadata.get("chunk_id", "")),
        "page_content": document.page_content,
        "metadata": dict(document.metadata),
        "rank": hit.rank,
        "score": hit.score,
        "distance": hit.distance,
    }


def deserialize_hit(payload: dict[str, Any]) -> RetrievalHit:
    document = Document(
        id=payload["id"],
        page_content=payload["page_content"],
        metadata=payload["metadata"],
    )
    return RetrievalHit(
        document=document,
        rank=int(payload["rank"]),
        score=float(payload["score"]),
        distance=payload["distance"],
    )


def intent_from_dict(payload: dict[str, Any]) -> QueryIntent:
    data = dict(payload)
    for key in ("requested_fields", "project_candidates", "diagnostics"):
        data[key] = tuple(data.get(key, ()))
    return QueryIntent(**data)

In [8]:
def normalize_query_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"normalized_query": normalize_lookup_text(state["question"])}


def parse_intent_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"intent": knowledge_base.parse_query(state["question"]).to_dict()}


def resolve_project_node(state: ReliableRAGState) -> dict[str, Any]:
    intent = state["intent"]
    return {"project_sources": list(intent.get("project_candidates", ())) }


def retrieve_candidates_node(state: ReliableRAGState) -> dict[str, Any]:
    hits = hybrid_candidates(state["normalized_query"])
    return {
        "candidate_hits": [serialize_hit(hit) for hit in hits],
        "retrieval_attempts": 1,
        "fallback_used": False,
        "retrieval_mode": "hybrid",
    }


def fallback_retrieve_node(state: ReliableRAGState) -> dict[str, Any]:
    """唯一 fallback：在现有候选之外补一轮 dense，随后必须结束。"""

    dense_hits = retrieve(vector_store, state["normalized_query"], k=12)
    merged: dict[str, dict[str, Any]] = {
        item["id"]: item for item in state.get("candidate_hits", [])
    }
    for hit in dense_hits:
        item = serialize_hit(hit)
        merged.setdefault(item["id"], item)
    return {
        "candidate_hits": list(merged.values()),
        "retrieval_attempts": state.get("retrieval_attempts", 0) + 1,
        "fallback_used": True,
        "retrieval_mode": "hybrid+dense_fallback",
    }


def locate_task_record_node(state: ReliableRAGState) -> dict[str, Any]:
    intent = intent_from_dict(state["intent"])
    hits = [deserialize_hit(item) for item in state.get("candidate_hits", [])]
    result = knowledge_base.lookup_intent(intent, retriever=lambda _: hits)
    payload = result.to_dict()
    payload["retrieval_calls"] = state.get("retrieval_attempts", 0)
    payload["diagnostics"]["graph_retrieval_mode"] = state.get(
        "retrieval_mode"
    )
    return {"lookup_result": payload}


def evidence_gate_node(state: ReliableRAGState) -> dict[str, Any]:
    # 唯一判定来源是共享 reliable pipeline，不在 graph 中复制规则。
    return {"evidence_status": state["lookup_result"]["status"]}


def route_after_gate(state: ReliableRAGState) -> str:
    status = state["evidence_status"]
    if status == "exact":
        return "format_answer"
    if status == "ambiguous":
        return "request_clarification"
    if status == "conflict":
        return "report_conflict"
    if status in {"not_found", "insufficient"} and state.get(
        "retrieval_attempts", 0
    ) < 2:
        return "fallback_retrieve"
    return "abstain"


def format_answer_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"answer": state["lookup_result"]["answer"]}


def request_clarification_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"answer": state["lookup_result"]["answer"]}


def abstain_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"answer": state["lookup_result"]["answer"]}


def report_conflict_node(state: ReliableRAGState) -> dict[str, Any]:
    return {"answer": state["lookup_result"]["answer"]}

In [9]:
builder = StateGraph(ReliableRAGState)
builder.add_node("normalize_query", normalize_query_node)
builder.add_node("parse_intent", parse_intent_node)
builder.add_node("resolve_project", resolve_project_node)
builder.add_node("retrieve_candidates", retrieve_candidates_node)
builder.add_node("locate_task_record", locate_task_record_node)
builder.add_node("evidence_gate", evidence_gate_node)
builder.add_node("fallback_retrieve", fallback_retrieve_node)
builder.add_node("format_answer", format_answer_node)
builder.add_node("request_clarification", request_clarification_node)
builder.add_node("abstain", abstain_node)
builder.add_node("report_conflict", report_conflict_node)

builder.add_edge(START, "normalize_query")
builder.add_edge("normalize_query", "parse_intent")
builder.add_edge("parse_intent", "resolve_project")
builder.add_edge("resolve_project", "retrieve_candidates")
builder.add_edge("retrieve_candidates", "locate_task_record")
builder.add_edge("locate_task_record", "evidence_gate")
builder.add_conditional_edges(
    "evidence_gate",
    route_after_gate,
    {
        "format_answer": "format_answer",
        "request_clarification": "request_clarification",
        "fallback_retrieve": "fallback_retrieve",
        "abstain": "abstain",
        "report_conflict": "report_conflict",
    },
)
builder.add_edge("fallback_retrieve", "locate_task_record")
for terminal in (
    "format_answer",
    "request_clarification",
    "abstain",
    "report_conflict",
):
    builder.add_edge(terminal, END)

checkpointer = InMemorySaver()
reliable_graph = builder.compile(checkpointer=checkpointer)
print("graph nodes:", sorted(reliable_graph.get_graph().nodes))

graph nodes: ['__end__', '__start__', 'abstain', 'evidence_gate', 'fallback_retrieve', 'format_answer', 'locate_task_record', 'normalize_query', 'parse_intent', 'report_conflict', 'request_clarification', 'resolve_project', 'retrieve_candidates']


## 6. 五类真实路径

这些问题都能从 v4 直接核验：普通成功、工程别名、跨工程同名歧义、不存在任务、父子层级。不存在任务会走满两次检索后拒答；歧义只检索一次并要求补工程名。

In [10]:
graph_cases = {
    "direct_success": "南溪输变电工程的地基基础施工在什么时间？",
    "project_alias": "珠海110kV黄金输变电工程的主体结构封顶什么时候完成？",
    "ambiguous": "施工准备什么时候完成？",
    "not_found": "南溪输变电工程的锅炉点火计划什么时候完成？",
    "parent_task": "110千伏节点计划中父任务试桩什么时候完成？",
}

demo_rows = []
for label, question in graph_cases.items():
    config = {"configurable": {"thread_id": f"langgraph-demo-{label}"}}
    final_state = reliable_graph.invoke({"question": question}, config=config)
    assert final_state["retrieval_attempts"] <= 2
    demo_rows.append(
        {
            "case": label,
            "status": final_state["evidence_status"],
            "retrievals": final_state["retrieval_attempts"],
            "fallback": final_state["fallback_used"],
            "answer": final_state["answer"].splitlines()[0],
        }
    )

print(json.dumps(demo_rows, ensure_ascii=False, indent=2))

[
  {
    "case": "direct_success",
    "status": "exact",
    "retrievals": 1,
    "fallback": false,
    "answer": "珠海110千伏南溪（旅游）输变电工程施工进度计划横道图中，“地基基础施工”计划开始2025年8月2日，计划完成2025年10月22日。"
  },
  {
    "case": "project_alias",
    "status": "exact",
    "retrievals": 1,
    "fallback": false,
    "answer": "珠海110千伏黄金输变电工程工程进度计划横道图中，“主体结构封顶”计划完成2024年10月24日。"
  },
  {
    "case": "ambiguous",
    "status": "ambiguous",
    "retrievals": 1,
    "fallback": false,
    "answer": "任务“施工准备”出现在多个工程文档中，请补充工程范围：110kV黄金输变电工程三级进度计划.txt、三级进度计划-土建.txt、三虎输变电工程三级进度计划土建部分.txt、珠海110千伏江湾输变电工程施工进度计划（202.txt"
  },
  {
    "case": "not_found",
    "status": "not_found",
    "retrievals": 2,
    "fallback": true,
    "answer": "知识库中没有找到任务“锅炉点火”的可验证记录。"
  },
  {
    "case": "parent_task",
    "status": "exact",
    "retrievals": 1,
    "fallback": false,
    "answer": "110千伏输变电工程节点计划中，“试桩”计划完成2028年2月29日。"
  }
]


## 7. Checkpoint 是线程内状态，不是长期知识记忆

`InMemorySaver` 按 `thread_id` 保存各 superstep 的快照。它适合 notebook 教学与调试，进程结束即消失。生产中的 durable execution 可换持久化 checkpointer，但当前 8 文档静态查询没有引入 SQLite/Postgres 的必要。

检索证据也不应写成跨线程长期记忆：事实来源是版本化语料与索引，长期记忆可能使旧日期绕过当前 Evidence Gate。

In [11]:
checkpoint_config = {
    "configurable": {"thread_id": "checkpoint-teaching-example"}
}
checkpoint_state = reliable_graph.invoke(
    {"question": graph_cases["project_alias"]},
    config=checkpoint_config,
)
current_snapshot = reliable_graph.get_state(checkpoint_config)
history = list(reliable_graph.get_state_history(checkpoint_config))

serialized = json.dumps(current_snapshot.values, ensure_ascii=False)
print(
    {
        "checkpoint_count": len(history),
        "current_status": current_snapshot.values["evidence_status"],
        "next_nodes": current_snapshot.next,
        "state_is_json_serializable": bool(serialized),
    }
)

{'checkpoint_count': 9, 'current_status': 'exact', 'next_nodes': (), 'state_is_json_serializable': True}


## 8. Graph 级评测：检索命中之外，还测澄清与拒答

正例仍报告 `source_hit@k/term_hit@k`；可靠性层另外报告工程路由、精确记录、字段完整、引用、歧义与拒答。13 条样例很小，只能作为本知识库的防回归，不代表开放域泛化能力。

In [12]:
def evaluate_graph(cases: list[dict[str, Any]]) -> dict[str, Any]:
    rows = []
    for index, case in enumerate(cases):
        config = {
            "configurable": {"thread_id": f"langgraph-eval-{index}"}
        }
        state = reliable_graph.invoke({"question": case["query"]}, config=config)
        result = state["lookup_result"]
        records = result["records"]
        expected_status = case.get("expected_status", "exact")
        expected_source = case.get("expected_source")
        expected_fields = case.get("expected_fields", {})
        status_ok = state["evidence_status"] == expected_status
        source_ok = expected_source is None or any(
            record["source_name"] == expected_source for record in records
        )
        fields_ok = all(
            any(str(record.get(key)) == str(value) for record in records)
            for key, value in expected_fields.items()
        )
        route_ok = expected_source is None or expected_source in state.get(
            "project_sources", []
        )
        citation_ok = (
            expected_status != "exact"
            or expected_source is None
            or expected_source in state["answer"]
        )
        rows.append(
            {
                "id": case.get("id"),
                "status_ok": status_ok,
                "source_ok": source_ok,
                "fields_ok": fields_ok,
                "route_ok": route_ok,
                "citation_ok": citation_ok,
                "retrievals": state["retrieval_attempts"],
                "fallback": state["fallback_used"],
                "expected_status": expected_status,
            }
        )

    exact_rows = [row for row in rows if row["expected_status"] == "exact"]
    ambiguity_rows = [
        row for row in rows if row["expected_status"] == "ambiguous"
    ]
    abstention_rows = [
        row
        for row in rows
        if row["expected_status"] in {"not_found", "insufficient", "conflict"}
    ]

    def rate(items: list[dict[str, Any]], key: str) -> float:
        return (
            sum(bool(item[key]) for item in items) / len(items) if items else 1.0
        )

    return {
        "cases": len(rows),
        "project_route_accuracy": rate(exact_rows, "route_ok"),
        "exact_record_match": sum(
            row["status_ok"] and row["source_ok"] for row in exact_rows
        )
        / len(exact_rows),
        "requested_field_completeness": rate(exact_rows, "fields_ok"),
        "citation_accuracy": rate(exact_rows, "citation_ok"),
        "ambiguity_detection": rate(ambiguity_rows, "status_ok"),
        "abstention_accuracy": rate(abstention_rows, "status_ok"),
        "average_retrievals": sum(row["retrievals"] for row in rows)
        / len(rows),
        "fallback_ratio": sum(row["fallback"] for row in rows) / len(rows),
        "all_pass_rate": sum(
            row["status_ok"] and row["source_ok"] and row["fields_ok"]
            for row in rows
        )
        / len(rows),
        "rows": rows,
    }


graph_report = evaluate_graph(all_cases)
metric_view = {key: value for key, value in graph_report.items() if key != "rows"}
print(json.dumps(metric_view, ensure_ascii=False, indent=2))
assert graph_report["all_pass_rate"] == 1.0
assert max(row["retrievals"] for row in graph_report["rows"]) <= 2

{
  "cases": 13,
  "project_route_accuracy": 1.0,
  "exact_record_match": 1.0,
  "requested_field_completeness": 1.0,
  "citation_accuracy": 1.0,
  "ambiguity_detection": 1.0,
  "abstention_accuracy": 1.0,
  "average_retrievals": 1.0769230769230769,
  "fallback_ratio": 0.07692307692307693,
  "all_pass_rate": 1.0
}


## 9. 工程结论

- **单跳事实**：直接 reliable pipeline 已经有明确输入、Evidence Gate、模板答案和引用，调用链更短，应作为默认路径。
- **LangGraph 适用点**：当需要可恢复状态、人工澄清、多个受控 fallback、跨步骤审计或后续审批时，显式 graph 更容易测试和维护。
- **本次 graph 没有使用 LLM**：规则足以识别当前工程与任务字段，LLM 不应被安排去“猜日期”。
- **准确率相同时也不是白做**：收益是每个分支、检索预算和失败原因可见，而不是高级框架自动改善检索。
- **代价**：graph 增加状态 schema、节点和 checkpoint 的维护复杂度；不存在任务会多做一次有界 dense fallback，延迟高于直接 pipeline。

对当前 8 份文档，推荐顺序仍是：可靠直接查询优先，确有多步骤控制需求再进入 LangGraph；不要把开放循环当作可靠性的替代品。